check autoloader-schemaeveloution and trigger data not loaed into target . cleared the checkpoint and retried also same 

In [0]:
dbutils.fs.put(source_path + "data_3.csv", "id,name,city,age\n4,Deepa,Pune,28", True)

In [0]:
source_path = "/Volumes/main/default/raw_data_volume/autoloader_csv/"
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/autoloader_1"
schema_path = "/Volumes/main/default/raw_data_volume/checkpoints/schema/"

dbutils.fs.mkdirs(source_path)
dbutils.fs.put(source_path + "data_1.csv", "id,name,city\n1,Aditya,Bangalore\n2,Arjun,Mumbai", True)
dbutils.fs.put(source_path + "data_2.csv", "id,name,city\n3,Vijay,Chennai", True)

from pyspark.sql.functions import current_timestamp
raw_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")                  # CSV specific: Use first row as header
    .option("cloudFiles.schemaLocation", schema_path) 
    .option("cloudFiles.inferColumnTypes", "true")    # Infers INT/Double instead of just String
    .load(source_path)
    .withColumn("ingestion_time", current_timestamp()) # Metadata for tracking
)

(raw_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")  # Enable schema evolution for new columns
    .trigger(availableNow=True)
    .toTable("main.default.autoloader_csv_demo")
)

In [0]:
%sql
-- Query the Delta table directly from the path
create table IF NOT EXISTS main.default.bronze_sensors_1
as SELECT * FROM delta.`/Volumes/main/default/raw_data_volume/delta_tables/bronze_sensors_2` ;

In [0]:
source_path = "/Volumes/main/default/raw_data_volume/exported_users_json/"
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/user_data_1"

# Auto Loader Stream
df_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json") # The format of the source files
  .option("cloudFiles.schemaLocation", checkpoint_path) # Where to store schema info
  .option("cloudFiles.inferColumnTypes", "true")
  .load(source_path)
)

(df_stream.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .option("mergeSchema", "true")
  .start("/Volumes/main/default/raw_data_volume/delta_tables/bronze_sensors_2")
)

In [0]:
# The S3 bucket path - replace with your actual S3 bucket name
#s3_source_path = "s3://your-bucket-name.s3.ap-southeast-2.amazonaws.com/orders/"
s3_source_path = "s3://databricksawsara/orders/"

# Use the existing raw_data_volume for checkpoints
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/s3_ingest"

# Streaming from S3 using cloudFiles
df_s3_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(s3_source_path)
)

# Writing to a Delta Table (table will be created automatically)
(df_s3_stream.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("main.default.bronze_sensors")
)

In [0]:
# Define the source and checkpoint paths using Unity Catalog Volumes
# Replace with your actual catalog, schema, and volume names
source_path = "/Volumes/main/default/raw_data_volume/exported_users_json/"
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/user_data"

# Auto Loader Stream
df_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json") # The format of the source files
  .option("cloudFiles.schemaLocation", checkpoint_path) # Where to store schema info
  .load(source_path)
)

# Write the stream to a Delta Table
(df_stream.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True) # "availableNow" is the modern version of 'once=True'
  .toTable("main.default.silver_users")
)